# Event Weights

Calculate concurrency, average uniqueness, return attribution, time decay, and normalized sample weights from the labeled AAPL events. Development and holdout are processed independently while preserving the established 64-column weighted-event schema.


## Process the Data


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing.event_weights import (
    apply_time_decay,
    compute_average_uniqueness_weights,
    compute_return_attribution_weights,
    count_concurrent_events,
)

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
event_path = event_dir / f"aapl_news_primary_model_{period}.parquet"
partition_path = event_dir / f"aapl_news_labeled_split_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
weighted_path = event_dir / f"aapl_news_modeling_weighted_{period}.parquet"

events = pd.read_parquet(event_path).sort_values("event_start", ignore_index=True)
partition_manifest = pd.read_parquet(partition_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")
events["event_start"] = pd.to_datetime(events["event_start"], utc=True)
events["event_end"] = pd.to_datetime(events["event_end"], utc=True)
partition_manifest["event_start"] = pd.to_datetime(partition_manifest["event_start"], utc=True)
close = dollar_bars.set_index("end")["close"].astype(float)


In [2]:
weight_tables = []
for partition in ["development", "holdout"]:
    partition_starts = partition_manifest.loc[partition_manifest["partition"].eq(partition), "event_start"]
    partition_events = events[events["event_start"].isin(partition_starts)].set_index("event_start")
    information_sets = partition_events["event_end"]
    concurrency = count_concurrent_events(close.index, information_sets, information_sets.index)
    uniqueness = compute_average_uniqueness_weights(information_sets, concurrency, information_sets.index)
    return_attribution = compute_return_attribution_weights(information_sets, concurrency, close, information_sets.index)
    positive_floor = return_attribution[return_attribution.gt(0)].min()
    if pd.isna(positive_floor):
        raise ValueError(f"{partition} return-attribution weights are all zero.")
    base_weight = return_attribution.clip(lower=positive_floor)
    time_decay = apply_time_decay(base_weight, clf_last_w=0.50)
    sample_weight = base_weight * time_decay
    sample_weight *= len(sample_weight) / sample_weight.sum()
    weight_tables.append(
        pd.DataFrame(
            {
                "average_uniqueness_weight": uniqueness,
                "return_attribution_weight": return_attribution,
                "time_decay_weight": time_decay,
                "sample_weight": sample_weight,
            }
        ).rename_axis("event_start").reset_index()
    )

weight_table = pd.concat(weight_tables, ignore_index=True).sort_values("event_start", ignore_index=True)
weighted_events = events.merge(weight_table, on="event_start", how="left", validate="one_to_one")
weight_columns = ["average_uniqueness_weight", "return_attribution_weight", "time_decay_weight", "sample_weight"]

assert weighted_events.shape[1] == 64
assert weighted_events["return_attribution_weight"].ge(0).all()
assert weighted_events[["average_uniqueness_weight", "time_decay_weight", "sample_weight"]].gt(0).all().all()
for partition in ["development", "holdout"]:
    starts = partition_manifest.loc[partition_manifest["partition"].eq(partition), "event_start"]
    partition_weights = weighted_events.loc[weighted_events["event_start"].isin(starts), "sample_weight"]
    assert abs(partition_weights.mean() - 1.0) < 1e-12

weighted_events.to_parquet(weighted_path, index=False)
print(weighted_path)


2026-08-19 23:09:59.500 | DEBUG    | src.data_preprocessing.event_weights:apply_time_decay:91 - Applied time decay with slope 0.14114377525442334 and intercept 0.5.


2026-08-19 23:09:59.536 | DEBUG    | src.data_preprocessing.event_weights:apply_time_decay:91 - Applied time decay with slope 1.1813359635731835 and intercept 0.5.


/Users/kwonjunhyuk9/Documents/financial-machine-learning/data/research_data/events/aapl_news_modeling_weighted_2025-01-01_2025-12-31.parquet


## Take a Quick Look at the Data Structure


In [3]:
development_starts = partition_manifest.loc[partition_manifest["partition"].eq("development"), "event_start"]
development_data = weighted_events[weighted_events["event_start"].isin(development_starts)]
development_data.head()

,event_start,symbol,event_end,vertical_barrier,target_return,raw_return,direction_label,mean_sentiment_score,fractionally_differenced_log_close,McClellan Oscillator,...,Keltner Channel Upper,Keltner Channel Middle,Keltner Channel Lower,Donchian Channel Upper,Donchian Channel Middle,Donchian Channel Lower,average_uniqueness_weight,return_attribution_weight,time_decay_weight,sample_weight
0,2025-01-02 15:00:32.232433+00:00,AAPL,2025-01-02 15:24:06.857548+00:00,2025-01-02 15:24:06.857548+00:00,0.007515,-0.005267,-1,0.090007,1.455318,2.6921,...,247.0100,246.6257,246.2414,246.99,246.570,246.15,1.000000,0.004875,0.500688,0.897596
1,2025-01-02 15:32:28.839475+00:00,AAPL,2025-01-02 15:49:31.568019+00:00,2025-01-02 15:49:31.568019+00:00,0.006929,-0.000061,-1,0.914739,1.453821,5.2764,...,245.8153,245.4296,245.0439,245.88,245.420,244.96,1.000000,0.001040,0.500835,0.191575
2,2025-01-02 16:48:23.973430+00:00,AAPL,2025-01-02 17:24:27.805471+00:00,2025-01-02 17:27:48.251810+00:00,0.002947,-0.003145,-1,-0.024872,1.452908,0.9699,...,245.1103,244.8782,244.6460,245.15,244.940,244.73,1.000000,0.003354,0.501308,0.618261
3,2025-01-02 17:35:50.819642+00:00,AAPL,2025-01-02 18:07:35.869987+00:00,2025-01-02 18:07:35.869987+00:00,0.003470,-0.000124,-1,0.052554,1.450704,-3.5400,...,243.4921,243.1607,242.8292,243.49,243.105,242.72,1.000000,0.000041,0.501314,0.007590
4,2025-01-03 15:00:13.717905+00:00,AAPL,2025-01-03 15:09:26.860772+00:00,2025-01-03 15:31:35.970761+00:00,0.003829,0.003910,1,0.851262,1.451799,0.0410,...,243.4799,242.8963,242.3127,243.15,242.765,242.38,0.527778,0.003063,0.501746,0.565107


In [4]:
development_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 978 entries, 0 to 977
Data columns (total 64 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   event_start                               978 non-null    datetime64[us, UTC]
 1   symbol                                    978 non-null    str                
 2   event_end                                 978 non-null    datetime64[us, UTC]
 3   vertical_barrier                          978 non-null    datetime64[us, UTC]
 4   target_return                             978 non-null    float64            
 5   raw_return                                978 non-null    float64            
 6   direction_label                           978 non-null    int8               
 7   mean_sentiment_score                      978 non-null    float64            
 8   fractionally_differenced_log_close        978 non-null    float64      

In [5]:
development_data["direction_label"].value_counts()

direction_label
 1    497
-1    481
Name: count, dtype: int64

In [6]:
development_data[weight_columns].describe()

,average_uniqueness_weight,return_attribution_weight,time_decay_weight,sample_weight
count,978.000000,978.000000,978.000000,978.000000
mean,0.812045,0.003622,0.780983,1.000000
std,0.232130,0.005227,0.153474,1.371672
min,0.173512,0.000000,0.500688,0.000115
25%,0.602124,0.001296,0.627368,0.358225
50%,0.990196,0.002439,0.815578,0.683755
75%,1.000000,0.004043,0.910675,1.148439
max,1.000000,0.059531,1.000000,17.026719


In [7]:
development_data[weight_columns].hist(figsize=(12, 8), bins=30)

array([[<Axes: title={'center': 'average_uniqueness_weight'}>,
        <Axes: title={'center': 'return_attribution_weight'}>],
       [<Axes: title={'center': 'time_decay_weight'}>,
        <Axes: title={'center': 'sample_weight'}>]], dtype=object)